# 00 Draw Validation Sample

Draws the stratified passage sample described in `PROTOCOL.md` and writes blind
annotation sheets for a single annotator.

**Run this where the full texts live — not in the public repository.**

| Output | Contains text? | Publish? |
|---|---|---|
| `sample_manifest.csv` | no — only ids and sentence indices | yes |
| `private_sheets/annotation_workbook.xlsx` | yes | **no** |

The workbook has empty label columns on purpose. Do not load the pipeline
predictions until annotation is finished: seeing the system output turns the
task into confirmation rather than annotation.

The design is sized for **three hours of work**: 40 passages, 200 sentences,
about 1,500 tokens, of which about 300 are tagged for part of speech.


In [ ]:
!pip install razdel openpyxl --quiet

## Parameters

`SOURCE_RUS_O` and `SOURCE_RUS_T` are the pipeline's own input tables
(semicolon-separated, `utf-8-sig`, with `id` and `text` columns) — the same
files notebooks 01 and 02 read.

Record `SEED` in the paper. If you redraw with a different seed, say so.


In [ ]:
import os

# --- inputs ---------------------------------------------------------------
BASE_PATH    = os.environ.get("KIDLIT_BASE", "./")
SOURCE_RUS_O = BASE_PATH + "kid_lit_100_ru.csv"
SOURCE_RUS_T = BASE_PATH + "kid_lit_100_foreign.csv"
METADATA     = "../data/metadata.csv"

# --- outputs --------------------------------------------------------------
WORK_DIR = "./private_sheets"   # contains text — never commit
OUT_DIR  = "."                  # manifest only — safe to commit

# --- design (see PROTOCOL.md, section 2) ----------------------------------
SEED = 20260807

# Equal numbers per subcorpus, not proportional to their size: the headline
# statistic is whether accuracy differs between RUS-O and RUS-T, and that
# comparison is most powerful with balanced groups. Verse is oversampled
# because line breaks without terminal punctuation are the hard case for
# sentence segmentation.
QUOTA = {("RUS-O", "prose"): 16, ("RUS-O", "verse"): 4,
         ("RUS-T", "prose"): 16, ("RUS-T", "verse"): 4}

PASSAGE_LEN = 5     # sentences per passage  -> 200 sentences
EDGE_MARGIN = 3     # skip this many sentences at either end of a text
POS_SENTS   = 1     # first N sentences of a passage go to the POS task

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# In Colab, mount Drive first and point KIDLIT_BASE at the folder there:
# from google.colab import drive; drive.mount("/content/drive")
# os.environ["KIDLIT_BASE"] = "/content/drive/MyDrive/.../"

## Load the source texts

`text_id` is built the same way as in `metadata.csv`, so the sample joins
cleanly to the released metadata.


In [ ]:
import random
import pandas as pd
from razdel import sentenize, tokenize

rng = random.Random(SEED)

def load_texts(paths):
    frames = []
    for origin, path in zip(("RUS-O", "RUS-T"), paths):
        d = pd.read_csv(path, sep=";", encoding="utf-8-sig")[["id", "text"]].copy()
        d["origin_type"] = origin
        d["text_id"] = [f"{origin}-{int(i):02d}" for i in d.id]
        frames.append(d.drop(columns="id"))
    return pd.concat(frames, ignore_index=True)

texts = load_texts([SOURCE_RUS_O, SOURCE_RUS_T])
meta  = pd.read_csv(METADATA)
df    = texts.merge(meta[["text_id", "genre"]], on="text_id", how="left")

print(df.shape)
df.groupby(["origin_type", "genre"]).size()

## Draw the passages

One passage per text: the quotas are smaller than every stratum's text count,
so all 40 passages come from 40 distinct books. The retry loop is a safety net
in case a stratum is ever shrunk below its quota.


In [ ]:
def draw(df, rng):
    manifest, sheets = [], []

    for (origin, genre), quota in QUOTA.items():
        pool = df[(df.origin_type == origin) & (df.genre == genre) & df.text.notna()]

        eligible = []
        for _, row in pool.iterrows():
            sents = [s.text.strip() for s in sentenize(str(row.text))]
            sents = [s for s in sents if s]
            if len(sents) >= PASSAGE_LEN + 2 * EDGE_MARGIN:
                eligible.append((row.text_id, sents))
        if not eligible:
            raise RuntimeError(f"no eligible texts in stratum {origin}/{genre}")

        chosen, used = [], {}
        while len(chosen) < quota:
            text_id, sents = eligible[len(chosen) % len(eligible)]
            lo, hi = EDGE_MARGIN, len(sents) - PASSAGE_LEN - EDGE_MARGIN
            taken = used.setdefault(text_id, set())
            for _ in range(200):
                start = rng.randint(lo, hi)
                span = set(range(start, start + PASSAGE_LEN))
                if not span & taken:
                    taken |= span
                    chosen.append((text_id, sents, start))
                    break
            else:
                raise RuntimeError(f"cannot place a passage in {text_id}")

        for text_id, sents, start in chosen:
            pid = f"P{len(manifest) + 1:03d}"
            manifest.append(dict(passage_id=pid, text_id=text_id,
                                 origin_type=origin, genre=genre,
                                 sent_index_from=start,
                                 sent_index_to=start + PASSAGE_LEN - 1))
            for k in range(PASSAGE_LEN):
                sheets.append(dict(passage_id=pid, text_id=text_id,
                                   origin_type=origin, genre=genre,
                                   sent_index=start + k,
                                   sentence=sents[start + k],
                                   in_pos_task=int(k < POS_SENTS)))
    return pd.DataFrame(manifest), pd.DataFrame(sheets)

manifest, sheets = draw(df, rng)
print(f"passages: {len(manifest)}   sentences: {len(sheets)}")
manifest.groupby(["origin_type", "genre"]).size()

## Write the public manifest

No text — ids, strata and sentence indices only. This is the file that lets a
reader verify the sampling design without seeing the books.


In [ ]:
manifest.to_csv(f"{OUT_DIR}/sample_manifest.csv", index=False,
                encoding="utf-8", lineterminator="\n")
manifest.head()

## Build the annotation workbook

Annotation happens in a spreadsheet, not in this notebook: a Colab runtime
times out, and three hours of work should not depend on a live kernel.

This writes `private_sheets/annotation_workbook.xlsx` with two tabs, ready to
be uploaded to Google Sheets (File → Import → Replace spreadsheet) or opened in
Excel. Dropdown validation, frozen headers and column widths are set up, so
every judgement is one keystroke and no tag can be mistyped.

**The workbook contains running text. Never commit it.**

### Tab `A_D_sentences` — one row per sentence (200 rows)

| Column | Fill with |
|---|---|
| `boundary_ok` | 1 if this really is exactly one sentence, else 0 |
| `error_type` | `missed`, `spurious`, `shifted` — only when `boundary_ok` = 0 |
| `coord_any` | 1 if any coordination is present, phrasal included |
| `coord_clause` | 1 if two or more clauses are coordinated |

### Tab `B_tokenisation` — one row per sentence (200 rows)

Tokens are shown joined by ` | `. Judge the whole sentence at once; only if
something is wrong do you name the offending positions.

| Column | Fill with |
|---|---|
| `all_correct` | 1 if every token in the row is right, else 0 |
| `bad_token_positions` | zero-based positions, comma-separated — only when `all_correct` = 0 |
| `n_tokens_gold` | prefilled with the pipeline count; correct it only when `all_correct` = 0 |

Part-of-speech annotation is built separately by `00b_pos_workbooks.ipynb`,
which shows the model's own tags for verification rather than asking you to
assign every tag from scratch.


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

# ---- assemble the three tables -------------------------------------------
ad = sheets[["passage_id", "text_id", "origin_type", "genre",
             "sent_index", "sentence"]].copy()
for col in ["boundary_ok", "error_type", "coord_any", "coord_clause", "note"]:
    ad[col] = None

bt_rows = []
for _, s in sheets.iterrows():
    toks = [t.text for t in tokenize(s.sentence)]
    bt_rows.append(dict(passage_id=s.passage_id, text_id=s.text_id,
                        origin_type=s.origin_type, sent_index=s.sent_index,
                        n_tokens=len(toks), tokenisation=" | ".join(toks),
                        all_correct=None, bad_token_positions=None,
                        n_tokens_gold=len(toks), note=None))

bt = pd.DataFrame(bt_rows)

# ---- write the workbook ---------------------------------------------------
HDR = PatternFill("solid", fgColor="DDDDDD")

def add_sheet(wb, name, frame, widths, wrap=(), validations=()):
    ws = wb.create_sheet(name)
    ws.append(list(frame.columns))
    for cell in ws[1]:
        cell.font, cell.fill = Font(bold=True), HDR
    for row in frame.itertuples(index=False):
        ws.append(["" if pd.isna(v) else v for v in row])
    ws.freeze_panes = "A2"
    for col, w in widths.items():
        ws.column_dimensions[get_column_letter(list(frame.columns).index(col) + 1)].width = w
    for col in wrap:
        idx = list(frame.columns).index(col) + 1
        for r in range(2, len(frame) + 2):
            ws.cell(r, idx).alignment = Alignment(wrap_text=True, vertical="top")
    for col, options in validations:
        idx = get_column_letter(list(frame.columns).index(col) + 1)
        dv = DataValidation(type="list", formula1='"' + ",".join(options) + '"',
                            allow_blank=True, showErrorMessage=True)
        ws.add_data_validation(dv)
        dv.add(f"{idx}2:{idx}{len(frame) + 1}")
    return ws

wb = Workbook()
wb.remove(wb.active)

add_sheet(wb, "A_D_sentences", ad,
          widths={"sentence": 90, "passage_id": 11, "text_id": 11,
                  "origin_type": 11, "genre": 8, "note": 30},
          wrap=("sentence",),
          validations=[("boundary_ok", ["1", "0"]),
                       ("error_type", ["missed", "spurious", "shifted"]),
                       ("coord_any", ["1", "0"]),
                       ("coord_clause", ["1", "0"])])

add_sheet(wb, "B_tokenisation", bt,
          widths={"tokenisation": 100, "passage_id": 11, "text_id": 11,
                  "origin_type": 11, "note": 30},
          wrap=("tokenisation",),
          validations=[("all_correct", ["1", "0"])])

path = f"{WORK_DIR}/annotation_workbook.xlsx"
wb.save(path)

print(f"{path}")
print(f"  A_D_sentences   {len(ad):>5} rows")
print(f"  B_tokenisation  {len(bt):>5} rows  ({int(bt.n_tokens.sum())} tokens behind them)")
print("\nPart-of-speech sheets are built separately: run 00b_pos_workbooks.ipynb.")

## Sanity check before annotating

Read a few passages. If any of them is a table of contents, a colophon, a
dedication or a run of page numbers, redraw with a different `SEED` and record
that you did.


In [ ]:
for pid in list(manifest.passage_id)[:3]:
    p = sheets[sheets.passage_id == pid]
    print(f"--- {pid}  {p.iloc[0].text_id}  {p.iloc[0].genre}")
    for _, r in p.iterrows():
        print("   ", r.sentence[:100])
    print()

## How to run the annotation

1. Upload `annotation_workbook.xlsx` to Google Drive and open it with Google
   Sheets, or work in it locally in Excel or LibreOffice. Sheets autosaves,
   which matters over a three-hour session.
2. Read section 4 of `PROTOCOL.md` and settle the decision rules for the
   ambiguous cases before touching the workbook.
3. Warm up on 10 sentences, check yourself against the rules, then re-do those
   10 in the normal pass.
4. Work tab by tab in order: `A_D_sentences`, then `B_tokenisation`. Do not
   interleave — each pass has its own rhythm.
5. Keep `hard_cases.md` open and log every case that took more than a few
   seconds of thought, with the decision and the reason.
6. Export each tab to CSV (File → Download → CSV) into `private_sheets/` as
   `A_D_sentences.csv` and `B_tokenisation.csv`.
7. Take a break, then run `00b_pos_workbooks.ipynb` for the part-of-speech
   task. Tagging in the second unbroken hour is measurably worse.

Do not open the pipeline predictions for tabs A, B and D at any point.
